# 2-2 回帰分析で要因と結果の関係を読む / Reading Relationships with Regression Analysis
『AIに頼んで動かす Python実務データ分析』第2章2節の参照用ノートブックです。 / Reference notebook for Chapter 2, Section 2.

`student_life.csv` をアップロード（またはサポートページから自動取得）し、`LANG` を選んで、すべてのセルを実行します。
Upload `student_life.csv`, choose `LANG`, and run all cells.

In [ ]:
LANG = "ja"   # "ja" / "en"
SAVE_FIGURES = True
BASE_URL = "https://raw.githubusercontent.com/YOUR_ACCOUNT/YOUR_REPO/main/data/"

In [ ]:
import os, subprocess, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
JP_FONT = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if LANG == "ja":
    if not os.path.exists(JP_FONT):
        subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True, check=True)
    fm.fontManager.addfont(JP_FONT)
    plt.rcParams["font.family"] = fm.FontProperties(fname=JP_FONT).get_name()
plt.rcParams.update({"font.size": 8, "axes.edgecolor": "black", "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 300})
FIG_W = 4.5
def save(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        os.makedirs(f"figures/{LANG}", exist_ok=True)
        fig.savefig(f"figures/{LANG}/{name}.png", bbox_inches="tight")
    plt.show()

In [ ]:
LABELS = {"ja": {"LifeSatisfaction": "生活満足度", "StudyHours": "学習時間", "AttendanceRate": "出席率",
                 "FriendCount": "友人数", "SleepHours": "睡眠時間", "ExerciseHours": "運動時間",
                 "MentalStress": "ストレス度", "InternetUse": "ネット利用時間", "PartTimeHours": "アルバイト時間"},
          "en": {"LifeSatisfaction": "Life satisfaction", "StudyHours": "Study hours", "AttendanceRate": "Attendance rate",
                 "FriendCount": "Friends", "SleepHours": "Sleep hours", "ExerciseHours": "Exercise hours",
                 "MentalStress": "Mental stress", "InternetUse": "Internet use", "PartTimeHours": "Part-time hours"}}[LANG]
TXT = {"ja": dict(std="標準化係数", ns="白抜き：有意でない（p ≥ 0.05）"),
       "en": dict(std="Standardized coefficient", ns="White bar: not significant (p ≥ 0.05)")}[LANG]

In [ ]:
import urllib.request
for f in ["student_life.csv", "student_life_codebook.csv"]:
    if not os.path.exists(f):
        try: urllib.request.urlretrieve(BASE_URL + f, f)
        except Exception as e: print(f, "をアップロードしてください / please upload", e)
df = pd.read_csv("student_life.csv")
print(df.shape)
df.describe().round(2).T

## 単回帰分析 / Simple regression (Figure 2-2-1)

In [ ]:
import statsmodels.api as sm
simple = sm.OLS(df.LifeSatisfaction, sm.add_constant(df.MentalStress)).fit()
print(simple.params.round(3).to_dict(), "R2 =", round(simple.rsquared, 3))
fig, ax = plt.subplots(figsize=(FIG_W, 2.6))
jit = np.random.default_rng(0).uniform(-0.2, 0.2, len(df))
ax.scatter(df.MentalStress + jit, df.LifeSatisfaction, s=5, color="#999999", lw=0)
xs = np.linspace(1, 10, 50)
ax.plot(xs, simple.params["const"] + simple.params["MentalStress"] * xs, "k-", lw=1.5)
ax.set_xlabel(LABELS["MentalStress"]); ax.set_ylabel(LABELS["LifeSatisfaction"]); ax.set_xticks(range(1, 11))
save(fig, "fig2-2-1_simple_regression")

## 重回帰分析 / Multiple regression

In [ ]:
X = df.drop(columns=["StudentID", "LifeSatisfaction"])
y = df.LifeSatisfaction
model = sm.OLS(y, sm.add_constant(X)).fit()
print(model.summary())

## VIF（多重共線性の確認） / Checking multicollinearity

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
Xc = sm.add_constant(X)
pd.Series({c: variance_inflation_factor(Xc.values, i) for i, c in enumerate(Xc.columns) if c != "const"}).round(2).rename("VIF")

## 標準化係数 / Standardized coefficients (Figure 2-2-2)

In [ ]:
Z = (X - X.mean()) / X.std()
yz = (y - y.mean()) / y.std()
std_model = sm.OLS(yz, sm.add_constant(Z)).fit()
coef = std_model.params.drop("const")
p = model.pvalues.drop("const")
order = coef.abs().sort_values().index
fig, ax = plt.subplots(figsize=(FIG_W, 2.6))
ax.barh([LABELS[c] for c in order], coef[order], color=["white" if p[c] >= 0.05 else "#666666" for c in order],
        edgecolor="black", lw=0.6)
for i, c in enumerate(order):
    v = coef[c]; v = 0.0 if abs(v) < 0.005 else v
    ax.text(v + (0.02 if v >= 0 else -0.02), i, f"{v:.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=7)
ax.axvline(0, color="black", lw=0.8); ax.set_xlim(-0.8, 0.4); ax.set_xlabel(TXT["std"])
ax.text(0.99, 1.02, TXT["ns"], transform=ax.transAxes, ha="right", va="bottom", fontsize=6.5)
save(fig, "fig2-2-2_standardized_coef")
pd.DataFrame({"coef": model.params.drop("const").round(4), "std_coef": coef.round(3), "p": p.round(3)})